# Exercise 2: Build a ReAct Agent by Hand (Ollama SDK)

In this exercise you build a ReAct agent **from scratch** with the Ollama Python SDK.

## What is a ReAct agent?

A ReAct agent combines **Reasoning** (thinking) with **Acting** (doing) in a loop:

```
User question
      |
+---> LLM thinks
|     |
|   Tool needed?  -- No --> Final answer
|     |
|    Yes
|     |
|   Run the tool
|     |
|   Result back to the LLM
+-----+
```

The agent decides **on its own** which tool it needs, calls it, gets the result, and keeps
thinking until it can give a final answer.

In Exercise 1 you watched this loop run step by step. Here you write it yourself.

## Provided files

| File | Contents |
|------|----------|
| `tools.py` | 3 tools: `web_search`, `calculator`, `read_file` + `tool_map` |
| `sample.txt` | Engineering formulas for testing `read_file` |

The model, parameters, and system prompt are set in the first code cell below, so everything
you might tune is in one place.

In [ ]:
import json
import ollama
from tools import tool_map

# ---- Configuration: everything you might tune lives here ----
MODEL          = "qwen3.5:4b"
NUM_CTX        = 32768   # size of the context window the model gets
TEMPERATURE    = 0.7     # 0 = deterministic, higher = more varied
MAX_ITERATIONS = 10      # safety cap on the reason-act loop

SYSTEM_PROMPT = """You are a thorough research and engineering assistant with access to tools.

Available tools:
- web_search(query): Search the web for current information
- calculator(expression): Calculate math expressions (supports +, -, *, /, **, sqrt, sin, cos, log, pi, e)
- read_file(filename): Read a local text file

Rules:
- Always use calculator for ANY arithmetic, never compute in your head.
- Always use web_search for facts you are not certain about.
- Always use read_file when the user references a file.
- Never fabricate tool results.
- Cite your sources when using web_search."""

print(f"Model: {MODEL}")
print(f"Tools: {list(tool_map.keys())}")

## Step 1: Tool definitions

Ollama needs JSON schemas to know which tools are available.
Each tool has a name, a description, and parameters.

This cell is complete: just run it and look at the structure.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Returns snippets with source URLs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query string"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression. Supports +, -, *, /, **, sqrt, sin, cos, log, pi, e.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a local text file. Only files in the project directory can be read.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The filename to read, e.g. 'sample.txt'"}
                },
                "required": ["filename"],
            },
        },
    }
]

print(f"{len(TOOLS)} tools defined: {[t['function']['name'] for t in TOOLS]}")

## Step 2: Build the ReAct loop

Now the core: the `run_agent` function. Here you fill in **4 TODOs**.

### How does `ollama.chat()` work?

```python
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Hello"}],
    tools=TOOLS,  # list of available tools
    options={"num_ctx": 32768, "temperature": 0.7}
)
```

The response is a dict:
- `response["message"]["content"]` — the answer text
- `response["message"]["tool_calls"]` — list of tool calls (if any)

A tool call looks like this:
```python
{"function": {"name": "calculator", "arguments": {"expression": "2 + 2"}}}
```

In [ ]:
def run_agent(user_input: str) -> str:
    """ReAct agent: reason, use tools, return the final answer."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]

    for i in range(MAX_ITERATIONS):
        print(f"\n--- Iteration {i+1} ---")

        # =============================================================
        # TODO 1: Call ollama.chat().
        #         Parameters: model=MODEL, messages=messages, tools=TOOLS,
        #                     options={"num_ctx": NUM_CTX, "temperature": TEMPERATURE}
        #
        # Input:  model, messages, tools, options
        # Output: response with response["message"]["content"]
        #         and response["message"]["tool_calls"]
        # =============================================================
        response = ...  # <-- implement here

        msg = response["message"]

        # =============================================================
        # TODO 2: Check whether msg contains tool calls.
        #
        # Input:  msg.get("tool_calls") -> list or None
        # Output: True if tool calls are present, False otherwise
        # =============================================================
        if ...:  # <-- condition here

            # Append the assistant message (with tool_calls) to the history
            messages.append(msg)

            for tc in msg["tool_calls"]:
                name = tc["function"]["name"]
                args = tc["function"]["arguments"]
                print(f"  Tool: {name}({json.dumps(args)})")

                # =====================================================
                # TODO 3: Run the tool.
                #
                # Input:  name = "calculator"
                #         args = {"expression": "(0.2 * 0.4**3) / 12"}
                # Output: result = "Result: 0.0010666666666666667"
                #
                #   a) Get the function from tool_map: tool_map[name]
                #   b) Call it with: func(**args)
                #   c) Append the result as {"role": "tool", "content": result}
                #      to messages
                # =====================================================
                func = ...    # <-- (a) get the function
                result = ...  # <-- (b) call the function
                messages.append(...)  # <-- (c) append the result

                print(f"  Result: {result[:200]}")

        else:
            # ==========================================================
            # TODO 4: No tool calls -> final answer.
            #
            # Input:  msg["content"] = "The result is 0.001067 m^4"
            # Output: return "The result is 0.001067 m^4"
            # ==========================================================
            final_answer = ...  # <-- implement here
            print(f"\n  Final answer: {final_answer[:300]}...")
            return final_answer

    return "Maximum iterations reached." 

## Step 3: Test the agent

Run the cells below to test your agent. If everything works, you should see sensible answers.

In [ ]:
# Test 1: Calculator
answer = run_agent("Calculate (0.2 * 0.4**3) / 12")
print(f"\nResult: {answer}")

In [ ]:
# Test 2: Read a file
answer = run_agent("Read the file sample.txt and list the formulas it contains.")
print(f"\nResult: {answer}")

In [ ]:
# Test 3: Web search
answer = run_agent("Search the web for the Young's modulus of structural steel S235.")
print(f"\nResult: {answer}")

## Done!

If all three tests work, you have built a working ReAct agent.

Continue with **Exercise 3**: there you build the same agent with LangChain and see how much
less code it takes, and how much transparency you trade away.